## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [2]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [ ]:
# from ner import evaluation

In [3]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [4]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [5]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann_sr = ner.ReadNERData()
wikiann_hr = ner.ReadNERData()

wikiann_sr_words, wikiann_sr_labels = wikiann_sr.read_dataset('wikiann', wikiann_label_map, lang='sr')
wikiann_hr_words, wikiann_hr_labels = wikiann_hr.read_dataset('wikiann', wikiann_label_map, lang='hr')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


...

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

# Evaluate model

In [8]:
alignment = {
'B-LOC': 'B-LOC',
'B-MISC': 'O',
'B-ORG': 'B-ORG',
'B-PER': 'B-PER',
'I-LOC': 'I-LOC',
'I-MISC': 'O',
'I-ORG': 'I-ORG',
'I-PER': 'I-PER',
'O': 'O',
 }

model_name = "classla/bcms-bertic-ner"
model_name_output = 'classla/bcms-bertic-ner'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

In [9]:
model_evaluation.model.config.id2label

{0: 'B-LOC',
 1: 'B-MISC',
 2: 'B-ORG',
 3: 'B-PER',
 4: 'I-LOC',
 5: 'I-MISC',
 6: 'I-ORG',
 7: 'I-PER',
 8: 'O'}

### wikiann - serbian

In [10]:
data_name = "wikiann_sr"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_sr_words, wikiann_sr_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [11]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.2306,0.1639,0.1916,3782
1,ORG,0.3883,0.2215,0.2821,3657
2,PER,0.5867,0.5151,0.5486,4139
3,micro,0.4236,0.3077,0.3564,11578
4,macro,0.4019,0.3002,0.3408,11578
5,weighted,0.4077,0.3077,0.3478,11578


In [12]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.3923,0.2382,0.2964,3782
1,B-ORG,0.5596,0.2926,0.3843,3657
2,B-PER,0.7360,0.5960,0.6587,4139
3,I-LOC,0.5197,0.0510,0.0929,9820
4,I-ORG,0.8245,0.2017,0.3241,7780
5,I-PER,0.8923,0.5047,0.6447,5962
6,O,0.6305,0.9936,0.7714,37048
7,accuracy,0.6418,72188,None,None
8,macro,0.6507,0.4111,0.4532,72188
9,weighted,0.6479,0.6418,0.5695,72188


### wikiann - croatian

In [13]:
data_name = "wikiann_hr"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_hr_words, wikiann_hr_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [14]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.7029,0.7746,0.7370,4862
1,ORG,0.7160,0.4605,0.5605,4100
2,PER,0.8115,0.8592,0.8347,4404
3,micro,0.7456,0.7061,0.7253,13366
4,macro,0.7434,0.6981,0.7107,13366
5,weighted,0.7427,0.7061,0.7150,13366


In [15]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.7769,0.8258,0.8006,4862
1,B-ORG,0.8317,0.5085,0.6311,4100
2,B-PER,0.8849,0.9253,0.9047,4404
3,I-LOC,0.6743,0.4099,0.5098,2818
4,I-ORG,0.9183,0.3981,0.5554,7285
5,I-PER,0.9492,0.8127,0.8756,5675
6,O,0.8773,0.9870,0.9289,57070
7,accuracy,0.8719,86214,None,None
8,macro,0.8446,0.6953,0.7437,86214
9,weighted,0.8714,0.8719,0.8575,86214
